# Expert Routing Analysis: Harmful vs Harmless Prompts

This notebook visualizes expert selection patterns in the OSS-20B MoE model for harmful and harmless prompts.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
%matplotlib inline

## Load Data

In [ ]:
# Load the expert routing data
data_file = "expert_routing_analysis/expert_routing_data.json"

print(f"Loading data from {data_file}...")
with open(data_file, 'r') as f:
    data = json.load(f)

print(f"\nModel: {data['model']}")
print(f"Number of layers: {data['num_layers']}")
print(f"Number of experts: {data['num_experts']}")
print(f"Experts per token: {data['experts_per_token']}")
print(f"Harmful prompts: {data['num_harmful']}")
print(f"Harmless prompts: {data['num_harmless']}")

## Aggregate Expert Counts

In [ ]:
def aggregate_expert_counts(results, label):
    """Aggregate expert counts across all layers and prompts."""
    layer_counts = {}
    
    if results:
        num_layers = len(results[0]["layer_routing"])
        
        for layer_idx in range(num_layers):
            layer_key = f"layer_{layer_idx}"
            all_experts = []
            
            for result in results:
                if layer_key in result["layer_routing"]:
                    top_experts = result["layer_routing"][layer_key]["top_expert"]
                    all_experts.extend(top_experts)
            
            layer_counts[layer_idx] = Counter(all_experts)
    
    # Aggregate across all layers
    total_counts = Counter()
    for layer_idx, counts in layer_counts.items():
        total_counts.update(counts)
    
    return total_counts, layer_counts

print("Aggregating expert usage data...")
harmful_counts, harmful_layer_counts = aggregate_expert_counts(data['harmful_results'], 'harmful')
harmless_counts, harmless_layer_counts = aggregate_expert_counts(data['harmless_results'], 'harmless')

num_experts = data['num_experts']
num_layers = len(harmful_layer_counts)

print(f"Aggregated data from {num_layers} MoE layers")
print(f"Total harmful expert selections: {sum(harmful_counts.values())}")
print(f"Total harmless expert selections: {sum(harmless_counts.values())}")

## 1. Side-by-Side Histograms

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))

# Normalize to percentages
harmful_total = sum(harmful_counts.values())
harmless_total = sum(harmless_counts.values())

experts = list(range(num_experts))
harmful_pcts = [100 * harmful_counts.get(i, 0) / harmful_total for i in experts]
harmless_pcts = [100 * harmless_counts.get(i, 0) / harmless_total for i in experts]

# Harmful distribution
ax1.bar(experts, harmful_pcts, color='#d62728', alpha=0.7)
ax1.set_xlabel('Expert ID', fontsize=12)
ax1.set_ylabel('Percentage (%)', fontsize=12)
ax1.set_title('Harmful Prompts: Expert Selection Distribution', fontsize=14, fontweight='bold')
ax1.set_xlim(-0.5, num_experts - 0.5)
ax1.grid(axis='y', alpha=0.3)

# Harmless distribution
ax2.bar(experts, harmless_pcts, color='#2ca02c', alpha=0.7)
ax2.set_xlabel('Expert ID', fontsize=12)
ax2.set_ylabel('Percentage (%)', fontsize=12)
ax2.set_title('Harmless Prompts: Expert Selection Distribution', fontsize=14, fontweight='bold')
ax2.set_xlim(-0.5, num_experts - 0.5)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Overlaid Bar Chart (Direct Comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))

experts = np.arange(num_experts)
width = 0.35

ax.bar(experts - width/2, harmful_pcts, width, label='Harmful', color='#d62728', alpha=0.8)
ax.bar(experts + width/2, harmless_pcts, width, label='Harmless', color='#2ca02c', alpha=0.8)

ax.set_xlabel('Expert ID', fontsize=14, fontweight='bold')
ax.set_ylabel('Percentage (%)', fontsize=14, fontweight='bold')
ax.set_title('Expert Selection Distribution: Harmful vs Harmless', fontsize=16, fontweight='bold')
ax.set_xticks(experts)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Difference Chart (Harmful - Harmless)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))

differences = [harmful_pcts[i] - harmless_pcts[i] for i in range(num_experts)]
colors = ['#d62728' if d > 0 else '#2ca02c' for d in differences]

ax.bar(experts, differences, color=colors, alpha=0.8)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

ax.set_xlabel('Expert ID', fontsize=14, fontweight='bold')
ax.set_ylabel('Difference in Percentage (Harmful - Harmless)', fontsize=14, fontweight='bold')
ax.set_title('Expert Usage Difference: Harmful vs Harmless\n(Red = More in Harmful, Green = More in Harmless)',
             fontsize=16, fontweight='bold')
ax.set_xticks(experts)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print top differences
diff_tuples = [(i, d) for i, d in enumerate(differences)]
diff_tuples.sort(key=lambda x: abs(x[1]), reverse=True)

print("\nTop 15 experts with largest differences:")
print("Expert ID | Difference (%) | More common in")
print("-" * 50)
for expert_id, diff in diff_tuples[:15]:
    label = "Harmful" if diff > 0 else "Harmless"
    print(f"  {expert_id:2d}      | {diff:+6.3f}        | {label}")

## 4. Top N Most-Used Experts

In [ ]:
# You can adjust this parameter
TOP_N = 15

fig, ax = plt.subplots(figsize=(14, 8))

# Get top experts by total usage
total_counts = Counter()
total_counts.update(harmful_counts)
total_counts.update(harmless_counts)
top_experts = [expert_id for expert_id, _ in total_counts.most_common(TOP_N)]

harmful_top_pcts = [harmful_pcts[i] for i in top_experts]
harmless_top_pcts = [harmless_pcts[i] for i in top_experts]

x = np.arange(len(top_experts))
width = 0.35

ax.barh(x - width/2, harmful_top_pcts, width, label='Harmful', color='#d62728', alpha=0.8)
ax.barh(x + width/2, harmless_top_pcts, width, label='Harmless', color='#2ca02c', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels([f'Expert {i}' for i in top_experts])
ax.set_xlabel('Percentage (%)', fontsize=14, fontweight='bold')
ax.set_title(f'Top {TOP_N} Most-Used Experts: Harmful vs Harmless', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Heatmap: Expert Usage by Layer

In [ ]:
# Create matrices for harmful and harmless
harmful_matrix = np.zeros((num_layers, num_experts))
harmless_matrix = np.zeros((num_layers, num_experts))

for layer_idx in range(num_layers):
    harmful_layer_total = sum(harmful_layer_counts[layer_idx].values())
    harmless_layer_total = sum(harmless_layer_counts[layer_idx].values())
    
    for expert_id in range(num_experts):
        harmful_matrix[layer_idx, expert_id] = 100 * harmful_layer_counts[layer_idx].get(expert_id, 0) / harmful_layer_total
        harmless_matrix[layer_idx, expert_id] = 100 * harmless_layer_counts[layer_idx].get(expert_id, 0) / harmless_layer_total

# Create difference matrix
diff_matrix = harmful_matrix - harmless_matrix

# Plot
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 8))

# Harmful heatmap
sns.heatmap(harmful_matrix, ax=ax1, cmap='Reds', cbar_kws={'label': 'Percentage (%)'})
ax1.set_xlabel('Expert ID', fontsize=12)
ax1.set_ylabel('Layer', fontsize=12)
ax1.set_title('Harmful Prompts: Expert Usage by Layer', fontsize=14, fontweight='bold')

# Harmless heatmap
sns.heatmap(harmless_matrix, ax=ax2, cmap='Greens', cbar_kws={'label': 'Percentage (%)'})
ax2.set_xlabel('Expert ID', fontsize=12)
ax2.set_ylabel('Layer', fontsize=12)
ax2.set_title('Harmless Prompts: Expert Usage by Layer', fontsize=14, fontweight='bold')

# Difference heatmap
sns.heatmap(diff_matrix, ax=ax3, cmap='RdYlGn_r', center=0, cbar_kws={'label': 'Difference (%)'})
ax3.set_xlabel('Expert ID', fontsize=12)
ax3.set_ylabel('Layer', fontsize=12)
ax3.set_title('Difference (Harmful - Harmless)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Statistical Summary

In [ ]:
from scipy.stats import entropy

# Calculate KL divergence
harmful_probs = np.array([harmful_counts.get(i, 0) for i in range(num_experts)])
harmless_probs = np.array([harmless_counts.get(i, 0) for i in range(num_experts)])

# Add-one smoothing
harmful_probs = (harmful_probs + 1) / (harmful_probs.sum() + num_experts)
harmless_probs = (harmless_probs + 1) / (harmless_probs.sum() + num_experts)

kl_div = np.sum(harmful_probs * np.log(harmful_probs / harmless_probs))

# Calculate entropy
harmful_entropy = entropy(harmful_probs)
harmless_entropy = entropy(harmless_probs)

print("Statistical Summary:")
print("="*50)
print(f"KL Divergence (Harmful || Harmless): {kl_div:.6f}")
print(f"Harmful entropy: {harmful_entropy:.4f} (max: {np.log(num_experts):.4f})")
print(f"Harmless entropy: {harmless_entropy:.4f} (max: {np.log(num_experts):.4f})")
print(f"\nActive experts (harmful): {len([c for c in harmful_counts.values() if c > 0])}/{num_experts}")
print(f"Active experts (harmless): {len([c for c in harmless_counts.values() if c > 0])}/{num_experts}")

## 7. Custom Analysis (Interactive)

Use the cells below to explore specific aspects of the data:

In [ ]:
# Example: Look at a specific layer
LAYER_TO_EXAMINE = 10  # Change this

layer_key = f"layer_{LAYER_TO_EXAMINE}"
harmful_layer = harmful_layer_counts[LAYER_TO_EXAMINE]
harmless_layer = harmless_layer_counts[LAYER_TO_EXAMINE]

harmful_layer_total = sum(harmful_layer.values())
harmless_layer_total = sum(harmless_layer.values())

layer_harmful_pcts = [100 * harmful_layer.get(i, 0) / harmful_layer_total for i in range(num_experts)]
layer_harmless_pcts = [100 * harmless_layer.get(i, 0) / harmless_layer_total for i in range(num_experts)]

fig, ax = plt.subplots(figsize=(18, 6))
x = np.arange(num_experts)
width = 0.35

ax.bar(x - width/2, layer_harmful_pcts, width, label='Harmful', color='#d62728', alpha=0.8)
ax.bar(x + width/2, layer_harmless_pcts, width, label='Harmless', color='#2ca02c', alpha=0.8)

ax.set_xlabel('Expert ID', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title(f'Layer {LAYER_TO_EXAMINE}: Expert Distribution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Example: Examine specific experts across layers
EXPERTS_TO_TRACK = [5, 9, 13, 30]  # Change this list

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

for expert_id in EXPERTS_TO_TRACK:
    harmful_layer_pcts = []
    harmless_layer_pcts = []
    
    for layer_idx in range(num_layers):
        h_total = sum(harmful_layer_counts[layer_idx].values())
        h_pct = 100 * harmful_layer_counts[layer_idx].get(expert_id, 0) / h_total
        harmful_layer_pcts.append(h_pct)
        
        ha_total = sum(harmless_layer_counts[layer_idx].values())
        ha_pct = 100 * harmless_layer_counts[layer_idx].get(expert_id, 0) / ha_total
        harmless_layer_pcts.append(ha_pct)
    
    ax1.plot(range(num_layers), harmful_layer_pcts, marker='o', label=f'Expert {expert_id}')
    ax2.plot(range(num_layers), harmless_layer_pcts, marker='o', label=f'Expert {expert_id}')

ax1.set_xlabel('Layer', fontsize=12)
ax1.set_ylabel('Percentage (%)', fontsize=12)
ax1.set_title('Harmful Prompts: Expert Usage Across Layers', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.set_xlabel('Layer', fontsize=12)
ax2.set_ylabel('Percentage (%)', fontsize=12)
ax2.set_title('Harmless Prompts: Expert Usage Across Layers', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()